In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
import joblib
import warnings
from pandas.errors import PerformanceWarning

warnings.filterwarnings("ignore", category = PerformanceWarning)

raw_df = pd.read_csv("weatherAUS.csv")

raw_df.dropna(subset=['RainToday', 'RainTomorrow'], inplace=True)

train_val_df, test_df = train_test_split(raw_df, test_size=0.2, random_state=42)
train_df, val_df = train_test_split(train_val_df, test_size=0.25, random_state=42)

input_cols = list(raw_df.columns)[1:-1]
target_col = 'RainTomorrow'

train_inputs = train_df[input_cols].copy()
train_targets = train_df[target_col].copy()
val_inputs = val_df[input_cols].copy()
val_targets = val_df[target_col].copy()
test_inputs = test_df[input_cols].copy()
test_targets = test_df[target_col].copy()

# Заповення пропусків
numeric_cols = train_inputs.select_dtypes(include=np.number).columns.tolist()
categorical_cols = train_inputs.select_dtypes('object').columns.tolist()

imputer = SimpleImputer(strategy = 'mean')
imputer_cat = SimpleImputer(strategy = 'most_frequent')

imputer.fit(train_inputs[numeric_cols])
imputer_cat.fit(train_inputs[categorical_cols])

train_inputs[numeric_cols] = imputer.transform(train_inputs[numeric_cols])
val_inputs[numeric_cols] = imputer.transform(val_inputs[numeric_cols])
test_inputs[numeric_cols] = imputer.transform(test_inputs[numeric_cols])

train_inputs[categorical_cols] = imputer_cat.transform(train_inputs[categorical_cols])
val_inputs[categorical_cols] = imputer_cat.transform(val_inputs[categorical_cols])
test_inputs[categorical_cols] = imputer_cat.transform(test_inputs[categorical_cols])

# Масштабування та кодування ознак
scaler = MinMaxScaler()

scaler.fit(train_inputs[numeric_cols])

train_inputs[numeric_cols] = scaler.transform(train_inputs[numeric_cols])
val_inputs[numeric_cols] = scaler.transform(val_inputs[numeric_cols])
test_inputs[numeric_cols] = scaler.transform(test_inputs[numeric_cols])

encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore', drop = 'if_binary')

encoder.fit(train_inputs[categorical_cols])

encoded_cols = list(encoder.get_feature_names_out(categorical_cols))

train_inputs[encoded_cols] = encoder.transform(train_inputs[categorical_cols])
val_inputs[encoded_cols] = encoder.transform(val_inputs[categorical_cols])
test_inputs[encoded_cols] = encoder.transform(test_inputs[categorical_cols])

feature_cols = numeric_cols + encoded_cols

X_train = train_inputs[feature_cols].copy()
X_val = val_inputs[feature_cols].copy()
X_test = test_inputs[feature_cols].copy()

# Створення екземпляра класифікатора випадкових лісів
clf = RandomForestClassifier(n_estimators=100)

# Тренування класифікатора на тренувальних даних
clf.fit(X_train, train_targets)

# Прогнозування на тестовому наборі
y_pred = clf.predict(X_test)

# Обчислення точності
accuracy = accuracy_score(test_targets, y_pred)
print(f"Точність: {accuracy:.2f}")

Точність: 0.85


In [2]:
joblib.dump(clf, "aussie_rain.joblib", compress=('zlib', 3))
joblib.dump(imputer, "imputer.joblib")
joblib.dump(imputer_cat, "imputer_cat.joblib")
joblib.dump(scaler, "scaler.joblib")
joblib.dump(encoder, "encoder.joblib")
joblib.dump(X_train.columns.tolist(),"feature_cols.joblib")

['feature_cols.joblib']

In [3]:
print(clf.classes_)

['No' 'Yes']


In [4]:
clf.predict_proba(X_test.iloc[[0]])[:,1]

array([0.28])